[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 Hard: Sliding Window Attention

Implement **Sliding Window Attention** — used in Longformer, Mistral, etc. for efficient long-context processing.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size).

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

In [2]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.3 MB/s eta 0:00:00


In [1]:
import torch
import math

In [17]:
# ✏️ YOUR IMPLEMENTATION HERE

def sliding_window_attention(Q, K, V, window_size):
    batch, seq, d = Q.shape
    assert K.shape == (batch, seq, d)
    assert V.shape == (batch, seq, d)
    #assert window_size >= 0

    scores = torch.einsum("bpd,bqd->bpq", Q, K) / math.sqrt(d)

    i = torch.arange(seq)
    i = i.reshape((seq, 1))
    j = torch.arange(seq)
    j = j.reshape((1, seq))
    valid = (i - j).abs() <= window_size #[a, b]
    #print(i.shape, j.shape, valid.shape, scores.shape)

    valid.reshape(1, seq, seq)

    scores = scores.masked_fill(~valid, float("-inf"))
    scores = torch.softmax(scores, dim=-1)

    return torch.einsum("bqs,bsd->bqd", scores, V)







In [18]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
print("Output shape:", out.shape)  # (1, 6, 8)

# window=0 should return V
out0 = sliding_window_attention(Q, K, V, window_size=0)
print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

torch.Size([6, 1]) torch.Size([1, 6]) torch.Size([6, 6]) torch.Size([1, 6, 6])
Output shape: torch.Size([1, 6, 8])
torch.Size([6, 1]) torch.Size([1, 6]) torch.Size([6, 6]) torch.Size([1, 6, 6])
window=0 == V? True


In [19]:
from torch_judge import check
check('sliding_window')


🧪 Testing: Sliding Window Attention (Hard)
──────────────────────────────────────────────────
torch.Size([8, 1]) torch.Size([1, 8]) torch.Size([8, 8]) torch.Size([2, 8, 8])
  ✅ [1/5] Output shape (17.9ms)
torch.Size([4, 1]) torch.Size([1, 4]) torch.Size([4, 4]) torch.Size([1, 4, 4])
  ✅ [2/5] window_size=0 — only sees itself (0.8ms)
torch.Size([6, 1]) torch.Size([1, 6]) torch.Size([6, 6]) torch.Size([2, 6, 6])
  ✅ [3/5] Large window equals full attention (6.6ms)
torch.Size([10, 1]) torch.Size([1, 10]) torch.Size([10, 10]) torch.Size([1, 10, 10])
torch.Size([10, 1]) torch.Size([1, 10]) torch.Size([10, 10]) torch.Size([1, 10, 10])
  ✅ [4/5] Distant tokens don't affect output (5.6ms)
torch.Size([4, 1]) torch.Size([1, 4]) torch.Size([4, 4]) torch.Size([2, 4, 4])
  ✅ [5/5] Gradient flow (22.2ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (53.1ms total)
  Progress saved. Run status() to see your dashboard.

